In [4]:
clauses = [
    ["~food(x)", "likes(John,x)"],
    ["food(Apple)"],
    ["food(Vegetables)"],
    ["~eats(x,y)", "killed(x)", "food(y)"],
    ["eats(Anil,Peanuts)"],
    ["alive(Anil)"],
    ["~eats(Anil,w)", "eats(Harry,w)"],
    ["killed(g)", "alive(g)"],
    ["~alive(k)", "~killed(k)"]
]

query = ["~likes(John,Peanuts)"]

variables = {"x", "y", "z", "k", "w", "g"}


def substitute(literal, subs):
    for var, value in subs.items():
        literal = literal.replace(var, value)
    return literal


def unify(atom1, atom2):
    if atom1 == atom2:
        return {}

    pred1, args1 = atom1.split("(")
    pred2, args2 = atom2.split("(")

    if pred1 != pred2:
        return None

    args1 = args1.rstrip(")").split(",")
    args2 = args2.rstrip(")").split(",")

    subs = {}

    for a, b in zip(args1, args2):
        a = substitute(a.strip(), subs)
        b = substitute(b.strip(), subs)

        if a == b:
            continue

        if a in variables:
            subs[a] = b
        elif b in variables:
            subs[b] = a
        else:
            return None

    return subs


def resolve(c1, c2):
    for l1 in c1:
        for l2 in c2:

            if l1.startswith("~") and not l2.startswith("~"):
                a = l1[1:]
                b = l2

            elif l2.startswith("~") and not l1.startswith("~"):
                a = l2[1:]
                b = l1

            else:
                continue

            subs = unify(a, b)

            if subs is not None:
                result = []

                for literal in c1:
                    if literal != l1:
                        result.append(substitute(literal, subs))

                for literal in c2:
                    if literal != l2:
                        result.append(substitute(literal, subs))

                return result, subs

    return None, None


print("KNOWLEDGE BASE")
print("-" * 45)

for i, clause in enumerate(clauses):
    print(f"C{i+1}: {' OR '.join(clause)}")

print("Negated Query:", query[0])


print("\nRESOLUTION STEPS")
print("-" * 45)

r1, s1 = resolve(clauses[0], query)

print("R1 =", r1)
print("Substitution:", s1)

r2, s2 = resolve(r1, clauses[3])

print("\nR2 =", r2)
print("Substitution:", s2)

r3, s3 = resolve(r2, clauses[4])

print("\nR3 =", r3)
print("Substitution:", s3)

r4, s4 = resolve(r3, clauses[8])

print("\nR4 =", r4)
print("Substitution:", s4)

r5, s5 = resolve(r4, clauses[5])

print("\nR5 =", r5)


print("\n\nPROOF TREE")
print("=" * 65)

if r5 == []:
    print("Empty clause {} obtained.")
    print("CONCLUSION: John likes peanuts is PROVED.")
else:
    print("Resolution failed.")


KNOWLEDGE BASE
---------------------------------------------
C1: ~food(x) OR likes(John,x)
C2: food(Apple)
C3: food(Vegetables)
C4: ~eats(x,y) OR killed(x) OR food(y)
C5: eats(Anil,Peanuts)
C6: alive(Anil)
C7: ~eats(Anil,w) OR eats(Harry,w)
C8: killed(g) OR alive(g)
C9: ~alive(k) OR ~killed(k)
Negated Query: ~likes(John,Peanuts)

RESOLUTION STEPS
---------------------------------------------
R1 = ['~food(Peanuts)']
Substitution: {'x': 'Peanuts'}

R2 = ['~eats(x,Peanuts)', 'killed(x)']
Substitution: {'y': 'Peanuts'}

R3 = ['killed(Anil)']
Substitution: {'x': 'Anil'}

R4 = ['~alive(Anil)']
Substitution: {'k': 'Anil'}

R5 = []


PROOF TREE
Empty clause {} obtained.
CONCLUSION: John likes peanuts is PROVED.
